# Мониторинг микросервисов: REST API + Prometheus + Grafana

В этом ноутбуке демонстрируется полный цикл наблюдения за контейнерным FastAPI-сервисом: сбор сырых метрик через `/metrics`, запуск искусственной нагрузки через `/load`, и визуализация временных рядов через Prometheus HTTP API.

## Что поднимается
- `fastapi` — порт 8000 (`/health`, `/metrics`, `/load`)
- `prometheus` — порт 9090
- `grafana` — порт 3000 (admin/admin, дашборд UID `microservice`)

Перед запуском ноутбука выполните: ```bash
docker compose up -d
# подождите ~10-15 секунд, пока Prometheus начнёт scrape
```


In [ ]:
import json
import time
from datetime import datetime, timedelta, timezone
import requests
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

print(f"requests=={requests.__version__}, pandas=={pd.__version__}")


## 1. Health-check сервиса


In [ ]:
r = requests.get("http://localhost:8000/health", timeout=5)
r.raise_for_status()
health = r.json()
print("HTTP:", r.status_code)
print(json.dumps(health, indent=2, ensure_ascii=False))
assert health["status"] == "ok", "Service is not ok"
print("OK: health-check passed")


## 2. Сырой scrape /metrics

Эндпоинт `/metrics` отдаёт данные в формате Prometheus exposition — текст с лейблами и типами. Соберём все метрики, относящиеся к нашему сервису.


In [ ]:
raw = requests.get("http://localhost:8000/metrics", timeout=5).text
print(f"Всего строк: {len(raw.splitlines())}")
interesting = [l for l in raw.splitlines()
               if l.startswith("app_") and not l.startswith("app_request_count_total{")]
for line in interesting[:30]:
    print(line)
print("...")
print(f"\nНайдено метрик приложения (без HTTP-счётчика): {len(interesting)}")
expected = ["app_cpu_usage_percent", "app_memory_usage_bytes",
            "app_disk_usage_bytes", "app_network_rx_bytes_total",
            "app_network_tx_bytes_total", "app_load_active"]
missing = [m for m in expected if not any(m in l for l in interesting)]
assert not missing, f"Не найдены метрики: {missing}"
print("OK: все ключевые метрики присутствуют")


## 3. Запуск нагрузки

Endpoint `POST /load?cpu_seconds=2&mem_mb=100&duration=15` запустит фоновую задачу: 2 секунды busy-loop CPU + 100 МБ аллокаций, всё автоматически освободится через 15 секунд.


In [ ]:
# POST http://localhost:8000/load — запускает фоновую нагрузку на сервис
r = requests.post(
    "http://localhost:8000/load",
    params={"cpu_seconds": 2, "mem_mb": 100, "duration": 15},
    timeout=5,
)
r.raise_for_status()
job = r.json()
print(json.dumps(job, indent=2, ensure_ascii=False))
assert r.status_code == 202, f"unexpected status {r.status_code}"
print("OK: нагрузка запущена")


## 4. Запрос временных рядов в Prometheus

Делаем паузу, чтобы Prometheus собрал свежие точки, и забираем данные за последние 2 минуты через `/api/v1/query_range`.


In [ ]:
time.sleep(10)  # дать Prometheus'у собрать несколько точек во время и после нагрузки
END = datetime.now(timezone.utc)
START = END - timedelta(minutes=2)
STEP = "2s"

def query_range(promql: str) -> pd.DataFrame:
    resp = requests.get(
        "http://localhost:9090/api/v1/query_range",
        params={"query": promql, "start": START.timestamp(),
                "end": END.timestamp(), "step": STEP},
        timeout=10,
    )
    resp.raise_for_status()
    body = resp.json()
    if body["status"] != "success":
        raise RuntimeError(f"Prometheus error: {body}")
    rows = []
    for series in body["data"]["result"]:
        metric = series["metric"]
        label = (metric.get("interface") or
                 metric.get("path") or
                 metric.get("__name__") or "")
        for ts, val in series["values"]:
            rows.append({"time": pd.to_datetime(ts, unit="s", utc=True),
                         "label": label,
                         "value": float(val)})
    return pd.DataFrame(rows)

cpu_df    = query_range("app_cpu_usage_percent")
mem_df    = query_range("app_memory_usage_bytes")
disk_df   = query_range('app_disk_usage_bytes{path="/"}')
rx_df     = query_range("rate(app_network_rx_bytes_total[10s])")
tx_df     = query_range("rate(app_network_tx_bytes_total[10s])")

for name, df in [("CPU", cpu_df), ("Memory", mem_df), ("Disk", disk_df),
                 ("RX", rx_df), ("TX", tx_df)]:
    print(f"{name}: {len(df)} точек, диапазон значений: "
          f"{df['value'].min() if len(df) else 'n/a'} .. {df['value'].max() if len(df) else 'n/a'}")


## 5. Визуализация

Строим сетку 2x2: CPU %, Memory MB, Disk GB, Network RX/TX KB/s.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle("Microservice Overview — last 2 min", fontsize=14)

# 1) CPU
ax = axes[0, 0]
if len(cpu_df):
    for label, sub in cpu_df.groupby("label"):
        ax.plot(sub["time"], sub["value"], label=label, linewidth=2)
ax.set_title("CPU Usage (%)")
ax.set_ylabel("%")
ax.set_ylim(0, max(100, cpu_df["value"].max() * 1.1 if len(cpu_df) else 100))
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", fontsize=8)

# 2) Memory MB
ax = axes[0, 1]
if len(mem_df):
    for label, sub in mem_df.groupby("label"):
        ax.plot(sub["time"], sub["value"] / (1024 * 1024), label=label, linewidth=2)
ax.set_title("Memory RSS (MB)")
ax.set_ylabel("MB")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", fontsize=8)

# 3) Disk GB
ax = axes[1, 0]
if len(disk_df):
    for label, sub in disk_df.groupby("label"):
        ax.plot(sub["time"], sub["value"] / (1024 ** 3), label=label, linewidth=2)
ax.set_title("Disk Usage — path /  (GB)")
ax.set_ylabel("GB")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", fontsize=8)

# 4) Network RX/TX KB/s
ax = axes[1, 1]
for df, label in [(rx_df, "RX"), (tx_df, "TX")]:
    if len(df):
        for lbl, sub in df.groupby("label"):
            ax.plot(sub["time"], sub["value"] / 1024,
                    label=f"{label} {lbl}", linewidth=2)
ax.set_title("Network Throughput (KB/s)")
ax.set_ylabel("KB/s")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", fontsize=8)

for ax in axes.flat:
    for label in ax.get_xticklabels():
        label.set_rotation(0)
        label.set_horizontalalignment("center")
    ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.show()


## 6. Grafana

Дашборд уже сконфигурирован через provisioning и доступен по адресу ниже. Откройте его в браузере, чтобы увидеть те же метрики в "production-grade" UI.


In [ ]:
grafana_url = "http://localhost:3000/d/microservice"
print(f"Grafana UI:       {grafana_url}")
print(f"Login:             admin / admin")
print(f"Prometheus UI:    http://localhost:9090")
print(f"FastAPI service:  http://localhost:8000/health")


In [ ]:
print("""
## Заключение

Мы выполнили полный цикл наблюдения:
1. Проверили liveness через REST.
2. Сняли сырые метрики Prometheus exposition.
3. Сгенерировали нагрузку и убедились, что метрики её отражают.
4. Запросили временные ряды из Prometheus и построили графики.
5. Указали путь к полноценному дашборду в Grafana.

Все компоненты работают изолированно в Docker Compose и могут быть расширены под боевую систему: добавление алертов в Alertmanager, долгосрочное хранение в Thanos/Mimir, логи в Loki.
""")
